# 清理 GWAS 结果：只保留 `.add.gz`

这个 notebook 做 4 件事：

1. 扫描 `per_pheno` 下所有 `*.glm.linear`
2. 检查是否已有对应的 `*.glm.linear.add` 或 `*.glm.linear.add.gz`
3. **只在对应 ADD 文件存在且非空时** 删除原始 `*.glm.linear`
4. 把 `*.glm.linear.add` 压缩成 `*.glm.linear.add.gz`

另外会顺手生成两个汇总：
- `cleanup_summary.tsv`：清理与压缩日志
- `add_minp_summary.tsv`：每个表型 ADD 文件的最小 P 和 lead SNP 汇总

## 建议
不建议只保留“显著文件”。  
更稳的做法是保留 **全部 `ADD.gz` + summary**，因为后面还可能要：
- 重新画 Manhattan / QQ
- 算 lambda
- 查 top hit
- 做 replication / annotation


In [ ]:

from pathlib import Path
import os
import gzip
import shutil
import time
import pandas as pd

# =========================
# 路径
# =========================
PROJECT_ROOT = Path("/data/work/CIMA_multiomics_regulation")
PER_PHENO_DIR = PROJECT_ROOT / "data/results/gwas_all/per_pheno"
OUT_DIR = PROJECT_ROOT / "data/results/gwas_all/_cleanup"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# =========================
# 开关
# =========================
DELETE_GLM_LINEAR = True      # 删除原始 *.glm.linear
GZIP_ADD = True               # 压缩 *.glm.linear.add -> *.glm.linear.add.gz
MAKE_MINP_SUMMARY = True      # 生成每个 phenotype 的最小P汇总
DRY_RUN = False               # True=只检查，不实际删除/压缩

# 并发压缩先不做，稳一点；单机顺序跑更安全
# 如果后面确认没问题，再改成 xargs -P 或 multiprocessing

assert PER_PHENO_DIR.exists(), f"目录不存在: {PER_PHENO_DIR}"

print("PROJECT_ROOT =", PROJECT_ROOT)
print("PER_PHENO_DIR =", PER_PHENO_DIR)
print("OUT_DIR =", OUT_DIR)
print("DELETE_GLM_LINEAR =", DELETE_GLM_LINEAR)
print("GZIP_ADD =", GZIP_ADD)
print("MAKE_MINP_SUMMARY =", MAKE_MINP_SUMMARY)
print("DRY_RUN =", DRY_RUN)


In [ ]:

from pathlib import Path
import pandas as pd

glm_files = sorted(PER_PHENO_DIR.rglob("*.glm.linear"))

records = []
for glm in glm_files:
    add_txt = Path(str(glm) + ".add")
    add_gz = Path(str(glm) + ".add.gz")
    phenotype = glm.parent.name

    rec = {
        "phenotype": phenotype,
        "glm_linear": str(glm),
        "glm_exists": glm.exists(),
        "glm_size": glm.stat().st_size if glm.exists() else 0,
        "add_txt": str(add_txt),
        "add_txt_exists": add_txt.exists(),
        "add_txt_size": add_txt.stat().st_size if add_txt.exists() else 0,
        "add_gz": str(add_gz),
        "add_gz_exists": add_gz.exists(),
        "add_gz_size": add_gz.stat().st_size if add_gz.exists() else 0,
    }
    rec["has_valid_add"] = (
        (rec["add_txt_exists"] and rec["add_txt_size"] > 0) or
        (rec["add_gz_exists"] and rec["add_gz_size"] > 0)
    )
    records.append(rec)

scan_df = pd.DataFrame(records)
scan_file = OUT_DIR / "scan_glm_add_status.tsv"
scan_df.to_csv(scan_file, sep="\t", index=False)

print("glm.linear file count:", len(scan_df))
print("with valid add/add.gz:", int(scan_df["has_valid_add"].sum()))
print("without valid add/add.gz:", int((~scan_df["has_valid_add"]).sum()))
print("saved:", scan_file)

display(scan_df.head())


In [ ]:

import gzip
import shutil
import os
import pandas as pd
from pathlib import Path

def gzip_file(src: Path, dst: Path):
    with open(src, "rb") as fin, gzip.open(dst, "wb", compresslevel=6) as fout:
        shutil.copyfileobj(fin, fout)

ops = []

for _, row in scan_df.iterrows():
    glm = Path(row["glm_linear"])
    add_txt = Path(row["add_txt"])
    add_gz = Path(row["add_gz"])
    phenotype = row["phenotype"]
    has_valid_add = bool(row["has_valid_add"])

    action = {
        "phenotype": phenotype,
        "glm_linear": str(glm),
        "add_txt": str(add_txt),
        "add_gz": str(add_gz),
        "has_valid_add": has_valid_add,
        "gz_created": False,
        "glm_deleted": False,
        "status": "skip",
        "note": ""
    }

    if not has_valid_add:
        action["status"] = "skip_no_valid_add"
        action["note"] = "对应 .add / .add.gz 不存在或为空，跳过删除"
        ops.append(action)
        continue

    # 1) 先压缩 add
    if GZIP_ADD and add_txt.exists() and add_txt.stat().st_size > 0 and (not add_gz.exists() or add_gz.stat().st_size == 0):
        if not DRY_RUN:
            gzip_file(add_txt, add_gz)
        action["gz_created"] = True

    # 2) 如果 add.gz 已存在且非空，可删除 add_txt
    if GZIP_ADD and add_txt.exists() and add_gz.exists() and add_gz.stat().st_size > 0:
        if not DRY_RUN:
            add_txt.unlink()
        action["note"] += " txt_add_removed_after_gzip;"

    # 3) 只有在 valid add 存在时，才删 glm.linear
    if DELETE_GLM_LINEAR and glm.exists():
        if not DRY_RUN:
            glm.unlink()
        action["glm_deleted"] = True

    action["status"] = "done"
    ops.append(action)

ops_df = pd.DataFrame(ops)
ops_file = OUT_DIR / "cleanup_summary.tsv"
ops_df.to_csv(ops_file, sep="\t", index=False)

print("done rows:", len(ops_df))
print("gz created:", int(ops_df["gz_created"].sum()))
print("glm deleted:", int(ops_df["glm_deleted"].sum()))
print("saved:", ops_file)

display(ops_df.head())


In [ ]:

# 压缩后再扫一遍，看看还剩什么
post_records = []
for p in sorted(PER_PHENO_DIR.rglob("*")):
    if p.is_file():
        suf = p.name
        if suf.endswith(".glm.linear") or suf.endswith(".glm.linear.add") or suf.endswith(".glm.linear.add.gz"):
            post_records.append({
                "path": str(p),
                "size": p.stat().st_size,
                "type": (
                    "glm.linear" if suf.endswith(".glm.linear")
                    else "add" if suf.endswith(".glm.linear.add")
                    else "add.gz"
                )
            })

post_df = pd.DataFrame(post_records)
post_file = OUT_DIR / "post_cleanup_files.tsv"
post_df.to_csv(post_file, sep="\t", index=False)

print(post_df["type"].value_counts(dropna=False))
print("saved:", post_file)
display(post_df.head())


In [ ]:

# 统计清理前后大小（基于扫描表 + 清理后文件表）
def human(n):
    if pd.isna(n):
        return "NA"
    n = float(n)
    for unit in ["B", "KB", "MB", "GB", "TB"]:
        if abs(n) < 1024:
            return f"{n:,.2f} {unit}"
        n /= 1024
    return f"{n:,.2f} PB"

before_glm = scan_df["glm_size"].sum()
before_add = scan_df["add_txt_size"].sum() + scan_df["add_gz_size"].sum()

after_glm = post_df.loc[post_df["type"] == "glm.linear", "size"].sum() if len(post_df) else 0
after_add = post_df.loc[post_df["type"].isin(["add", "add.gz"]), "size"].sum() if len(post_df) else 0

size_summary = pd.DataFrame([
    {"stage": "before", "glm_bytes": before_glm, "add_bytes": before_add, "total_bytes": before_glm + before_add},
    {"stage": "after",  "glm_bytes": after_glm,  "add_bytes": after_add,  "total_bytes": after_glm + after_add},
])

size_summary["glm_human"] = size_summary["glm_bytes"].map(human)
size_summary["add_human"] = size_summary["add_bytes"].map(human)
size_summary["total_human"] = size_summary["total_bytes"].map(human)

size_file = OUT_DIR / "size_summary.tsv"
size_summary.to_csv(size_file, sep="\t", index=False)

display(size_summary)
print("saved:", size_file)


In [ ]:

# 生成每个 phenotype 的最小 P 汇总
# 说明：这里只扫 ADD / ADD.GZ，保留全量结果更稳，不建议只保留显著文件
if MAKE_MINP_SUMMARY:
    import pandas as pd
    from pathlib import Path

    add_files = sorted(PER_PHENO_DIR.rglob("*.glm.linear.add.gz"))
    if len(add_files) == 0:
        add_files = sorted(PER_PHENO_DIR.rglob("*.glm.linear.add"))

    minp_records = []

    for f in add_files:
        phenotype = f.parent.name
        try:
            df = pd.read_csv(f, sep="\t", low_memory=False)
            if "P" not in df.columns:
                minp_records.append({
                    "phenotype": phenotype,
                    "file": str(f),
                    "status": "failed_no_P_column"
                })
                continue

            df["P"] = pd.to_numeric(df["P"], errors="coerce")
            df = df.dropna(subset=["P"]).copy()
            df = df[df["P"] > 0].copy()

            if df.empty:
                minp_records.append({
                    "phenotype": phenotype,
                    "file": str(f),
                    "status": "empty_after_filter"
                })
                continue

            lead = df.loc[df["P"].idxmin()].copy()

            minp_records.append({
                "phenotype": phenotype,
                "file": str(f),
                "status": "ok",
                "lead_id": lead["ID"] if "ID" in lead.index else None,
                "lead_chr": lead["#CHROM"] if "#CHROM" in lead.index else None,
                "lead_pos": lead["POS"] if "POS" in lead.index else None,
                "lead_p": lead["P"],
                "lead_beta": lead["BETA"] if "BETA" in lead.index else None,
                "lead_se": lead["SE"] if "SE" in lead.index else None,
                "n_rows": len(df),
                "suggestive_hit": bool((df["P"] < 1e-5).any()),
                "gws_hit": bool((df["P"] < 5e-8).any()),
            })
        except Exception as e:
            minp_records.append({
                "phenotype": phenotype,
                "file": str(f),
                "status": f"failed:{type(e).__name__}:{e}"
            })

    minp_df = pd.DataFrame(minp_records).sort_values(
        by=["status", "lead_p"], ascending=[True, True], na_position="last"
    )
    minp_file = OUT_DIR / "add_minp_summary.tsv"
    minp_df.to_csv(minp_file, sep="\t", index=False)

    print("saved:", minp_file)
    display(minp_df.head(20))


## 我对“要不要先筛显著再打包”的建议

不建议把非显著 ADD 直接丢掉。原因很简单：

- 你后面可能还要重画 QQ / Manhattan
- 可能还要算 lambda / 检查 inflation
- 可能要回头看某个 phenotype 的 top 区域
- replication 时也不一定只看 genome-wide significant

所以更合理的是：

**保留全部 `ADD.gz`，再额外产出一个显著汇总表。**

这样空间省了，分析灵活性也还在。
